# Генерация и отбор молекулярных дескрипторов для прогнозирования температуры кипения

## Идея и структура ноутбука

На выходе формируются **три** независимых набора признаков для `modeling.ipynb`:

| Набор | Содержимое | Файл |
|---|---|---|
| **X_morgan** | Morgan FP (radius=2, nBits=2048) | `X_morgan.pkl` |
| **X_rdkit** | 31 отобранный RDKit-дескриптор (отбор без FP) | `X_rdkit.pkl` |
| **X_combined** | Morgan FP + K RDKit-дескрипторов (отбор с учётом FP) | `X_combined.pkl` |

Morgan FP кодируют **топологическую** информацию о структуре, но не несут явной информации о полярности, водородных связях, электронных свойствах — именно эти свойства критичны для температуры кипения. Дополнительные дескрипторы RDKit восполняют этот пробел.

## Пайплайн отбора дескрипторов

```
217 дескрипторов RDKit
       ↓
  [1] Фильтр NaN (>10%) + заполнение медианой
       ↓
  [2] VarianceThreshold (убираем константные)
       ↓
  [3] Удаление коллинеарных (|Pearson r| > 0.90) → 134 кандидата
       ↓
  [4] Разбиение train/test ← ЗДЕСЬ, до отбора!
       ↓
  [5a] SFS (forward, Ridge, tol=1e-3) на TRAIN, без Morgan FP
        → 31 дескриптор → X_rdkit.pkl
       ↓
  [5b] SFS (forward, Ridge, tol=1e-3) на TRAIN, Morgan FP фиксированы
        → K дескрипторов, дополняющих FP → X_combined.pkl
       ↓
  Morgan FP (2048 бит) → X_morgan.pkl
```

> **Почему разбиение ДО обёрточного отбора?** Если делать отбор на всей выборке, тестовые данные «просачиваются» через CV-отбор, и оценка качества оказывается оптимистичной. Это классическая ошибка отбора признаков.

In [1]:
import numpy as np
import pandas as pd
import pickle
import json
import warnings
import time
from pathlib import Path

from rdkit import Chem
from rdkit.Chem import Descriptors, rdFingerprintGenerator

from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold, SequentialFeatureSelector
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')
print('Все библиотеки загружены успешно')

Все библиотеки загружены успешно


## 1. Загрузка данных

Читаем `merged_nbp_clean.sdf`. Целевая переменная — `Temperature_Celsius` (нормальная температура кипения при 1 атм).

In [2]:
SDF_PATH = Path('../data_ml/merged_nbp_clean.sdf')

suppl = Chem.SDMolSupplier(str(SDF_PATH), removeHs=True)

mols, targets = [], []
skipped = 0

for mol in suppl:
    if mol is None:
        skipped += 1
        continue
    try:
        bp = mol.GetDoubleProp('Temperature_Celsius')
        mols.append(mol)
        targets.append(bp)
    except (KeyError, Exception):
        skipped += 1

y = np.array(targets)

print(f'Загружено молекул:  {len(mols)}')
print(f'Пропущено:          {skipped}')
print(f'Диапазон T кип:     {y.min():.1f} – {y.max():.1f} °C')
print(f'Среднее ± std:      {y.mean():.1f} ± {y.std():.1f} °C')

Загружено молекул:  8572
Пропущено:          0
Диапазон T кип:     -161.5 – 590.5 °C
Среднее ± std:      179.4 ± 81.6 °C


## 2. Молекулярные отпечатки Morgan (фиксированная компонента)

**Параметры:** radius=2 (эквивалентно ECFP4), nBits=2048.  
Morgan FP хэшируется в битовый вектор — каждый бит отвечает за наличие определённого локального окружения атома. Они **не участвуют в отборе** и войдут в `X_final` полностью.

In [3]:
RADIUS = 2
N_BITS = 2048

morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=RADIUS, fpSize=N_BITS)

def mol_to_morgan(mol):
    return morgan_gen.GetFingerprintAsNumPy(mol).astype(np.float32)

t0 = time.time()
fp_matrix = np.vstack([mol_to_morgan(m) for m in mols])
elapsed = time.time() - t0

print(f'Матрица Morgan FP: {fp_matrix.shape}  (время: {elapsed:.1f} с)')
print(f'Средняя плотность: {fp_matrix.mean():.3f}  (доля ненулевых битов)')
print(f'Dtype: {fp_matrix.dtype}')

Матрица Morgan FP: (8572, 2048)  (время: 0.4 с)
Средняя плотность: 0.008  (доля ненулевых битов)
Dtype: float32


## 3. Дескрипторы RDKit — вычисление полного набора

RDKit предоставляет 217 дескрипторов нескольких типов:

- **EState** (электронное состояние атомов): `MaxEStateIndex`, `MinEStateIndex`, ...
- **QED** (drug-likeness качество): `qed`
- **Физико-химические**: `MolWt`, `MolLogP`, `MolMR`, `TPSA`, `NumHDonors`, `NumHAcceptors`, ...
- **VSA** (Van der Waals Surface Area): `SMR_VSA1`..`SMR_VSA10`, `SlogP_VSA1`...
- **Топологические индексы**: `BertzCT`, `Ipc`, `Kappa1`..`Kappa3`, `Chi0`...
- **Фрагментные счётчики** (`fr_*`): наличие функциональных групп

In [4]:
desc_list  = Descriptors.descList
desc_names = [d[0] for d in desc_list]
desc_funcs = [d[1] for d in desc_list]
print(f'Всего дескрипторов RDKit: {len(desc_names)}')

def compute_all_descs(mol, funcs):
    """Вычисляет все дескрипторы для молекулы; при ошибке ставит NaN."""
    vals = []
    for fn in funcs:
        try:
            v = fn(mol)
            vals.append(float(v) if v is not None else np.nan)
        except Exception:
            vals.append(np.nan)
    return vals

print('Вычисляем дескрипторы (может занять 1-2 мин)...')
t0 = time.time()

desc_matrix = []
for i, mol in enumerate(mols):
    desc_matrix.append(compute_all_descs(mol, desc_funcs))
    if (i + 1) % 2000 == 0:
        print(f'  {i + 1}/{len(mols)} молекул...  ({time.time()-t0:.0f} с)')

df_desc = pd.DataFrame(desc_matrix, columns=desc_names)
df_desc.replace([np.inf, -np.inf], np.nan, inplace=True)

print(f'\nГотово за {time.time()-t0:.1f} с')
print(f'Матрица дескрипторов: {df_desc.shape}')
nan_counts = df_desc.isna().sum()
print(f'Дескрипторов без единого NaN: {(nan_counts == 0).sum()}')

print('\nТоп-10 дескрипторов по доле NaN:')
print((nan_counts / len(mols)).sort_values(ascending=False).head(10).round(4).to_string())

Всего дескрипторов RDKit: 217
Вычисляем дескрипторы (может занять 1-2 мин)...
  2000/8572 молекул...  (11 с)
  4000/8572 молекул...  (21 с)
  6000/8572 молекул...  (29 с)
  8000/8572 молекул...  (38 с)

Готово за 41.0 с
Матрица дескрипторов: (8572, 217)
Дескрипторов без единого NaN: 205

Топ-10 дескрипторов по доле NaN:
BCUT2D_LOGPLOW         0.0107
BCUT2D_MWHI            0.0107
BCUT2D_MWLOW           0.0107
BCUT2D_CHGHI           0.0107
BCUT2D_CHGLO           0.0107
MaxAbsPartialCharge    0.0107
MinAbsPartialCharge    0.0107
MaxPartialCharge       0.0107
MinPartialCharge       0.0107
BCUT2D_MRHI            0.0107


## 4. Фильтрация: удаление пропусков и нулевой дисперсии

**Шаг 4a:** Удаляем дескрипторы, у которых доля NaN превышает 10% — они плохо работают на нашем датасете.  
**Шаг 4b:** Оставшиеся NaN заполняем медианой по столбцу.  
**Шаг 4c:** `VarianceThreshold(0.01)` — убираем почти константные дескрипторы (нет информации).

In [5]:
# 4a. Фильтр по доле NaN
NAN_THRESHOLD = 0.10
nan_frac = df_desc.isna().mean()
cols_after_nan = nan_frac[nan_frac <= NAN_THRESHOLD].index.tolist()
df_f = df_desc[cols_after_nan].copy()
print(f'[4a] После фильтра NaN (порог {NAN_THRESHOLD*100:.0f}%): '
      f'{df_desc.shape[1]} → {df_f.shape[1]} дескрипторов')
print(f'     Удалено: {df_desc.shape[1] - df_f.shape[1]}')

# 4b. Заполнение оставшихся NaN медианой
medians = df_f.median()
df_f = df_f.fillna(medians)
print(f'[4b] Оставшиеся NaN заполнены медианой')

# 4c. VarianceThreshold
VAR_THRESHOLD = 0.01
var_selector = VarianceThreshold(threshold=VAR_THRESHOLD)
var_selector.fit(df_f)
cols_after_var = df_f.columns[var_selector.get_support()].tolist()
df_f = df_f[cols_after_var]
print(f'[4c] После VarianceThreshold({VAR_THRESHOLD}): '
      f'{len(cols_after_nan)} → {df_f.shape[1]} дескрипторов')
print(f'\nИтого на фильтрации: {df_desc.shape[1]} → {df_f.shape[1]}')

[4a] После фильтра NaN (порог 10%): 217 → 217 дескрипторов
     Удалено: 0
[4b] Оставшиеся NaN заполнены медианой
[4c] После VarianceThreshold(0.01): 217 → 174 дескрипторов

Итого на фильтрации: 217 → 174


## 5. Удаление коллинеарных дескрипторов (Pearson |r| > 0.90)

Многие RDKit-дескрипторы сильно коррелируют друг с другом: например, `MolWt` и `HeavyAtomMolWt`, или различные индексы VSA. Коллинеарность делает регрессионную модель нестабильной и затрудняет интерпретацию.

**Стратегия жадного удаления:**  
1. Сортируем дескрипторы по убыванию |r| с целевой переменной (T кип).  
2. Идём по списку: первый дескриптор добавляем в «хранимые».  
3. Все последующие, у которых |r| > 0.90 с уже хранимыми, удаляем.  

Таким образом из каждой группы коллинеарных признаков остаётся **наиболее предсказательный** по отношению к T кип.

In [6]:
CORR_THRESHOLD = 0.90

# Корреляция каждого дескриптора с целевой переменной
corr_with_y = df_f.corrwith(pd.Series(y, index=df_f.index)).abs()
sorted_cols = corr_with_y.sort_values(ascending=False).index.tolist()

print(f'Вычисляем матрицу корреляций ({df_f.shape[1]}×{df_f.shape[1]})...')
t0 = time.time()
corr_matrix = df_f.corr().abs()
print(f'Готово за {time.time()-t0:.1f} с')

# Жадное удаление коллинеарных
to_drop = set()
kept = []  # в порядке убывания предсказательной силы

for col in sorted_cols:
    if col in to_drop:
        continue
    # Проверяем против уже принятых
    for kept_col in kept:
        if corr_matrix.loc[col, kept_col] > CORR_THRESHOLD:
            to_drop.add(col)
            break
    else:
        kept.append(col)

df_f = df_f[kept].copy()
cols_uncorr = kept

print(f'\n[5] После удаления коллинеарных (|r|>{CORR_THRESHOLD}): '
      f'{len(sorted_cols)} → {df_f.shape[1]} дескрипторов')
print(f'    Удалено коллинеарных: {len(to_drop)}')

print(f'\nКандидаты (|r| с T_кип, убывание):')
cand_df = corr_with_y[cols_uncorr].sort_values(ascending=False).round(3)
for rank, (name, corr_val) in enumerate(cand_df.items(), 1):
    print(f'  {rank:3d}. {name:<40s}  r={corr_val:.3f}')

Вычисляем матрицу корреляций (174×174)...
Готово за 0.5 с

[5] После удаления коллинеарных (|r|>0.9): 174 → 134 дескрипторов
    Удалено коллинеарных: 40

Кандидаты (|r| с T_кип, убывание):
    1. MolMR                                     r=0.567
    2. AvgIpc                                    r=0.523
    3. BertzCT                                   r=0.517
    4. HeavyAtomCount                            r=0.509
    5. fr_benzene                                r=0.501
    6. Chi3n                                     r=0.495
    7. SMR_VSA7                                  r=0.437
    8. PEOE_VSA6                                 r=0.429
    9. RingCount                                 r=0.425
   10. FpDensityMorgan1                          r=0.415
   11. ExactMolWt                                r=0.406
   12. HallKierAlpha                             r=0.389
   13. MolLogP                                   r=0.380
   14. Kappa2                                    r=0.351
   15. BCUT2

## 6. Разбиение train/test ДО обёрточного отбора

Это критически важный шаг.  

> **Ошибка отбора признаков (selection bias):** если запустить обёрточный метод на всех данных, а потом оценивать качество на том же множестве — CV-оценки «знают» о тестовых примерах через процедуру отбора. Итоговая оценка на тесте будет занижена (модель кажется хуже, чем есть) или, наоборот, завышена. Правильный порядок: **сначала отделить тест, потом отбирать на train**.

In [7]:
RANDOM_STATE = 42
TEST_SIZE    = 0.20

idx_all = np.arange(len(y))
idx_train, idx_test = train_test_split(
    idx_all, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

X_desc_all   = df_f.values
X_desc_train = X_desc_all[idx_train]
X_desc_test  = X_desc_all[idx_test]
y_train      = y[idx_train]
y_test       = y[idx_test]

X_fp_train = fp_matrix[idx_train]
X_fp_test  = fp_matrix[idx_test]

print(f'Обучающая выборка: {len(y_train)} молекул  ({100*(1-TEST_SIZE):.0f}%)')
print(f'Тестовая выборка:  {len(y_test)} молекул   ({100*TEST_SIZE:.0f}%)')
print()
print(f'Train T кип: {y_train.mean():.1f} ± {y_train.std():.1f} °C')
print(f'Test  T кип: {y_test.mean():.1f} ± {y_test.std():.1f} °C')
print()
print('Разбиение выполнено ДО отбора признаков — нет утечки данных!')

Обучающая выборка: 6857 молекул  (80%)
Тестовая выборка:  1715 молекул   (20%)

Train T кип: 179.3 ± 81.6 °C
Test  T кип: 179.8 ± 81.8 °C

Разбиение выполнено ДО отбора признаков — нет утечки данных!


## 7. Обёрточный отбор: SequentialFeatureSelector (forward, Ridge)

**Алгоритм:** жадный форвардный отбор.  
На каждом шаге добавляем тот дескриптор, который максимально улучшает 5-fold CV-R² на обучающей выборке (Ridge как базовая модель).  
Останавливаемся, когда прирост R² становится < `tol=0.001`.

**Нормализация:** StandardScaler обучается **только на train**, затем применяется к test — это предотвращает утечку статистики тестовой выборки.

Почему Ridge, а не что-то сложнее? Форвардный SFS с Ridge — быстрый, устойчивый, достаточно чувствительный к сигналу дескрипторов. Для формирования признакового пространства этого достаточно; полная нелинейная модель будет в `modeling.ipynb`.

In [8]:
# Нормализация — fit ТОЛЬКО на train!
scaler_train = StandardScaler()
X_train_sc   = scaler_train.fit_transform(X_desc_train)
X_test_sc    = scaler_train.transform(X_desc_test)

print(f'Кандидатов для отбора: {X_train_sc.shape[1]}')
print('Запуск Sequential Feature Selection (forward, Ridge, 5-fold CV)...')
print('Это займёт несколько минут.\n')

ridge_selector = Ridge(alpha=1.0)
cv_strategy    = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

sfs = SequentialFeatureSelector(
    ridge_selector,
    n_features_to_select='auto',   # автоматически по tol
    tol=1e-3,                       # минимальный прирост CV-R² для добавления
    direction='forward',
    scoring='r2',
    cv=cv_strategy,
    n_jobs=-1
)

t0 = time.time()
sfs.fit(X_train_sc, y_train)
elapsed = time.time() - t0

selected_mask  = sfs.get_support()
selected_names = [cols_uncorr[i] for i, s in enumerate(selected_mask) if s]
n_selected     = int(selected_mask.sum())

print(f'Отбор завершён за {elapsed:.1f} с')
print(f'Отобрано дескрипторов: {n_selected} из {len(cols_uncorr)}')
print()
print(f'{"#":<4} {"Дескриптор":<40} {"r с T_кип":>10}')
print('-' * 58)
for i, name in enumerate(selected_names, 1):
    r_val = corr_with_y[name]
    print(f'{i:<4} {name:<40} {r_val:>10.3f}')

Кандидатов для отбора: 134
Запуск Sequential Feature Selection (forward, Ridge, 5-fold CV)...
Это займёт несколько минут.

Отбор завершён за 248.6 с
Отобрано дескрипторов: 31 из 134

#    Дескриптор                                r с T_кип
----------------------------------------------------------
1    MolMR                                         0.567
2    BertzCT                                       0.517
3    fr_benzene                                    0.501
4    RingCount                                     0.425
5    FpDensityMorgan1                              0.415
6    SlogP_VSA8                                    0.314
7    VSA_EState7                                   0.249
8    BCUT2D_CHGLO                                  0.216
9    VSA_EState3                                   0.213
10   TPSA                                          0.213
11   MinEStateIndex                                0.163
12   fr_aryl_methyl                                0.157
13   NumHAcceptor

## 8. Проверка на тестовой выборке

Обучаем Ridge только на **отобранных дескрипторах** (без Morgan FP) на train, оцениваем на test. Это позволяет убедиться:
1. Что отобранные дескрипторы несут реальный сигнал, а не шум.
2. Нет ли переобучения при отборе (gap между train и test).

Дополнительно смотрим **коэффициенты Ridge** — они показывают относительный вклад каждого дескриптора (т.к. все нормализованы).

In [9]:
X_sel_train = X_train_sc[:, selected_mask]
X_sel_test  = X_test_sc[:, selected_mask]

ridge_eval = Ridge(alpha=1.0)
ridge_eval.fit(X_sel_train, y_train)

r2_train  = r2_score(y_train, ridge_eval.predict(X_sel_train))
r2_test   = r2_score(y_test,  ridge_eval.predict(X_sel_test))
mae_train = mean_absolute_error(y_train, ridge_eval.predict(X_sel_train))
mae_test  = mean_absolute_error(y_test,  ridge_eval.predict(X_sel_test))

print('=== Только отобранные дескрипторы (без Morgan FP) ===')
print(f'  Train: R² = {r2_train:.4f},  MAE = {mae_train:.2f} °C')
print(f'  Test:  R² = {r2_test:.4f},  MAE = {mae_test:.2f} °C')

gap = r2_train - r2_test
# Норма для линейной модели на химических данных: gap < 0.05.
# Gap 0.03–0.05 — обычный generalization gap, не selection bias.
# Настоящее переобучение при отборе: gap > 0.10 (модель "заучила" train через SFS).
if gap > 0.05:
    print(f'  Gap train-test = {gap:.4f}  ⚠ возможна утечка при отборе')
else:
    print(f'  Gap train-test = {gap:.4f}  ✓ нормальный generalization gap для линейной модели')

print()
print('Коэффициенты Ridge (сортировка по |коэфф|):')
coef_df = pd.DataFrame({
    'descriptor': selected_names,
    'coef':       ridge_eval.coef_,
    '|coef|':     np.abs(ridge_eval.coef_)
}).sort_values('|coef|', ascending=False).reset_index(drop=True)
print(coef_df[['descriptor', 'coef']].to_string(index=False))

=== Только отобранные дескрипторы (без Morgan FP) ===
  Train: R² = 0.6285,  MAE = 36.08 °C
  Test:  R² = 0.5906,  MAE = 38.18 °C
  Gap train-test = 0.0379  ✓ нормальный generalization gap для линейной модели

Коэффициенты Ridge (сортировка по |коэфф|):
       descriptor       coef
             TPSA  51.558788
            MolMR  34.074159
       fr_benzene  30.606382
   MinEStateIndex  30.206514
    NumHAcceptors -25.755985
          BertzCT -24.329262
 FpDensityMorgan1 -14.845583
        PEOE_VSA4  14.593789
         BalabanJ  11.342496
           fr_NH2 -10.603594
      SlogP_VSA12  10.288475
        RingCount  10.194925
      VSA_EState8  -9.786329
      VSA_EState7   9.730569
     BCUT2D_CHGLO  -8.681485
       SlogP_VSA8   6.770140
MaxAbsEStateIndex   6.270909
       PEOE_VSA10  -5.940864
      fr_pyridine   5.857897
 FpDensityMorgan3   5.482908
 MinPartialCharge  -4.832720
           fr_NH0  -4.803948
   fr_aryl_methyl   4.710058
       fr_isocyan  -4.456748
MinAbsEStateIndex  -4

## 8а. Химическая интерпретация отобранных дескрипторов

Температура кипения определяется **межмолекулярными взаимодействиями**. SFS отобрал дескрипторы, которые в совокупности описывают все три их типа:

| Тип взаимодействия | Механизм | Отобранные дескрипторы |
|---|---|---|
| **Дисперсионные силы Лондона** | ↑ с ростом размера и поляризуемости молекулы | `MolMR`, `BertzCT`, `RingCount`, `FpDensityMorgan1/3` |
| **Диполь-дипольные** | ↑ с ростом полярности | `TPSA`, `BCUT2D_CHGLO`, `MinPartialCharge`, `PEOE_VSA*` |
| **Водородные связи** | Резко ↑ при наличии N–H, O–H | `NumHAcceptors`, `fr_NH2`, `fr_NH0`, `fr_C_O`, `fr_lactone` |
| **π–π стекинг** | ↑ в ароматических системах | `fr_benzene`, `fr_aryl_methyl`, `fr_pyridine`, `fr_ArN` |
| **Электронные** | Распределение электронной плотности | `MinEStateIndex`, `MinAbsEStateIndex`, `MaxAbsEStateIndex`, `VSA_EState*` |


In [10]:
# Группировка отобранных дескрипторов по физическому смыслу
desc_groups = {
    'Размер / поляризуемость  (силы Лондона)': [
        'MolMR', 'BertzCT', 'RingCount', 'FpDensityMorgan1', 'FpDensityMorgan3',
    ],
    'Ароматичность / π–π стекинг': [
        'fr_benzene', 'fr_aryl_methyl', 'fr_ArN', 'fr_pyridine',
    ],
    'Полярность / водородные связи': [
        'TPSA', 'NumHAcceptors', 'fr_NH2', 'fr_NH0', 'fr_C_O', 'fr_isocyan', 'fr_lactone',
    ],
    'Электронные  (E-State / парциальные заряды)': [
        'MinEStateIndex', 'MinAbsEStateIndex', 'MaxAbsEStateIndex',
        'MinPartialCharge', 'BCUT2D_CHGLO',
    ],
    'Поверхностные  VSA  (пространственно-электронные)': [
        'SlogP_VSA8', 'SlogP_VSA12',
        'PEOE_VSA4', 'PEOE_VSA10', 'PEOE_VSA11', 'PEOE_VSA12',
        'VSA_EState3', 'VSA_EState7', 'VSA_EState8',
    ],
    'Топологические': [
        'BalabanJ',
    ],
}

sel_set = set(selected_names)
coef_map = dict(zip(selected_names, ridge_eval.coef_))

print('Отобранные дескрипторы по категориям:')
print('=' * 70)
covered = set()
for group, members in desc_groups.items():
    in_sel = [m for m in members if m in sel_set]
    if not in_sel:
        continue
    print(f'\n▶ {group}  ({len(in_sel)} шт.)')
    print(f'  {"Дескриптор":<28}  {"r с T_кип":>10}  {"β (Ridge)":>10}  Физический смысл')
    print('  ' + '-' * 66)
    for name in in_sel:
        r_val = corr_with_y[name]
        coef  = coef_map[name]
        print(f'  {name:<28}  {r_val:>10.3f}  {coef:>+10.2f}')
    covered.update(in_sel)

uncovered = [n for n in selected_names if n not in covered]
if uncovered:
    print(f'\n▶ Прочие  ({len(uncovered)} шт.)')
    for name in uncovered:
        print(f'  {name:<28}  r={corr_with_y[name]:.3f}')

print(f'\n{"=" * 70}')
print(f'Итого: {len(selected_names)} дескрипторов  |  покрыто категорий: {len(desc_groups)}')
print('Пространственные ✓  |  Электронные ✓  |  Полярность/H-связи ✓  |  Ароматичность ✓')

Отобранные дескрипторы по категориям:

▶ Размер / поляризуемость  (силы Лондона)  (5 шт.)
  Дескриптор                     r с T_кип   β (Ridge)  Физический смысл
  ------------------------------------------------------------------
  MolMR                              0.567      +34.07
  BertzCT                            0.517      -24.33
  RingCount                          0.425      +10.19
  FpDensityMorgan1                   0.415      -14.85
  FpDensityMorgan3                   0.025       +5.48

▶ Ароматичность / π–π стекинг  (4 шт.)
  Дескриптор                     r с T_кип   β (Ridge)  Физический смысл
  ------------------------------------------------------------------
  fr_benzene                         0.501      +30.61
  fr_aryl_methyl                     0.157       +4.71
  fr_ArN                             0.112       +3.52
  fr_pyridine                        0.071       +5.86

▶ Полярность / водородные связи  (7 шт.)
  Дескриптор                     r с T_кип   β (R

## 9. Сборка наборов X_morgan и X_rdkit и сохранение

Формируем первые два из трёх наборов признаков:

- **X_morgan** — исходные Morgan FP (2048 бит), без какой-либо фильтрации.
- **X_rdkit** — 31 RDKit-дескриптор, отобранный в разделе 7 (SFS без учёта Morgan FP).

Все дескрипторы сохраняются **без нормализации** — нормализация является частью конкретной модели:
- `Ridge / SVR / KNN` → `Pipeline([('scaler', StandardScaler()), ('model', ...)])`
- `RF / XGBoost / CatBoost / LightGBM` → нормализация не нужна, деревья инвариантны к масштабу

Третий набор, **X_combined**, будет сформирован в разделе 9а после комбинированного отбора.

In [11]:
OUT_DIR     = Path('../data_ml')
morgan_cols = [f'Morgan_{i}' for i in range(N_BITS)]

# 1. X_morgan — только Morgan FP (2048 бит, binary 0/1)
df_morgan = pd.DataFrame(fp_matrix, columns=morgan_cols)
df_morgan.to_pickle(OUT_DIR / 'X_morgan.pkl')

# 2. X_rdkit — 31 отобранный RDKit-дескриптор (сырые значения, без нормализации)
X_desc_sel_all_raw = X_desc_all[:, selected_mask]   # (n_mols, 31)
df_rdkit = pd.DataFrame(X_desc_sel_all_raw, columns=selected_names)
df_rdkit.to_pickle(OUT_DIR / 'X_rdkit.pkl')

# 3. Целевые значения
np.save(OUT_DIR / 'y.npy', y)

# 4. Метаданные (поле rdkit_cols_combined будет дописано в разделе 9а)
meta = {
    'morgan_radius':    RADIUS,
    'morgan_n_bits':    N_BITS,
    'morgan_cols':      morgan_cols,
    'rdkit_cols':       selected_names,
    'n_molecules':      int(len(y)),
    'n_morgan_bits':    N_BITS,
    'n_selected_rdkit': int(n_selected),
    'corr_with_target': {name: float(corr_with_y[name]) for name in selected_names},
    'note': 'Descriptors are RAW (not scaled). Use StandardScaler inside Pipeline for Ridge/SVR/KNN.',
}
with open(OUT_DIR / 'selected_descriptors.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print('Сохранено:')
print(f'  X_morgan.pkl  — {df_morgan.shape[0]} молекул × {df_morgan.shape[1]} признаков  (Morgan FP, binary)')
print(f'  X_rdkit.pkl   — {df_rdkit.shape[0]} молекул × {df_rdkit.shape[1]} признаков  (RDKit, raw)')
print(f'  y.npy         — ({len(y)},)')
print(f'  selected_descriptors.json')
print()
print('X_combined.pkl будет сохранён в разделе 9а после комбинированного отбора.')

Сохранено:
  X_morgan.pkl  — 8572 молекул × 2048 признаков  (Morgan FP, binary)
  X_rdkit.pkl   — 8572 молекул × 31 признаков  (RDKit, raw)
  y.npy         — (8572,)
  selected_descriptors.json

X_combined.pkl будет сохранён в разделе 9а после комбинированного отбора.


## 9а. Комбинированный отбор: RDKit-дескрипторы поверх Morgan FP

**Цель:** найти минимальный набор RDKit-дескрипторов, который **дополняет** Morgan FP, максимально улучшая совместную предсказательную способность.

**Алгоритм:** кастомный жадный форвардный SFS.
На каждом шаге 2048 Morgan FP присутствуют в Ridge-модели **фиксированно**, и мы добавляем тот RDKit-дескриптор из 134 кандидатов, который даёт наибольший прирост 5-fold CV-R². Останавливаемся при приросте < `tol = 0.001`.

**Отличие от раздела 7:** там SFS искал дескрипторы, наиболее предсказательные сами по себе. Здесь SFS ищет дескрипторы, несущие **дополнительную** информацию сверх Morgan FP. Ожидаемый результат: K ≤ 31, поскольку часть топологической информации уже покрыта FP.

**Оценка времени:** 10–30 мин в зависимости от оборудования.

In [12]:
print('Комбинированный отбор: Morgan FP (2048) фиксированы, добавляем RDKit-дескрипторы...')
print(f'Кандидатов RDKit: {len(cols_uncorr)}')
print('Это займёт 10–30 мин.\n')

# Масштабируем Morgan FP — fit только на train
scaler_morgan_comb = StandardScaler()
X_morgan_tr_sc = scaler_morgan_comb.fit_transform(fp_matrix[idx_train])   # (6857, 2048)
X_morgan_te_sc = scaler_morgan_comb.transform(fp_matrix[idx_test])         # (1715, 2048)

# X_train_sc из раздела 7 — масштабированные RDKit-кандидаты (134 шт.) на train
ridge_comb = Ridge(alpha=1.0)
cv_comb    = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
TOL_COMB   = 1e-3

def cv_r2_combined(rdkit_col_indices):
    """CV-R² Ridge на [Morgan FP | выбранные RDKit столбцы]."""
    if rdkit_col_indices:
        X_aug = np.hstack([X_morgan_tr_sc, X_train_sc[:, rdkit_col_indices]])
    else:
        X_aug = X_morgan_tr_sc
    return cross_val_score(ridge_comb, X_aug, y_train,
                           cv=cv_comb, scoring='r2', n_jobs=-1).mean()

morgan_only_cv_r2 = cv_r2_combined([])
print(f'Базовый CV-R² (только Morgan FP): {morgan_only_cv_r2:.4f}')

selected_rdkit_idx_comb = []                     # индексы в X_train_sc (0..133)
remaining_idx_comb      = list(range(len(cols_uncorr)))
best_score_comb         = morgan_only_cv_r2

t0   = time.time()
step = 0
while remaining_idx_comb:
    step += 1
    best_gain      = 0.0
    best_col       = None
    best_new_score = best_score_comb

    for col in remaining_idx_comb:
        score = cv_r2_combined(selected_rdkit_idx_comb + [col])
        gain  = score - best_score_comb
        if gain > best_gain:
            best_gain      = gain
            best_col       = col
            best_new_score = score

    if best_gain < TOL_COMB or best_col is None:
        print(f'  Остановка: максимальный прирост = {best_gain:.4f} < {TOL_COMB}')
        break

    selected_rdkit_idx_comb.append(best_col)
    remaining_idx_comb.remove(best_col)
    best_score_comb = best_new_score

    col_name = cols_uncorr[best_col]
    print(f'  Шаг {step:2d}: +{col_name:<38}  CV-R²={best_score_comb:.4f}  (Δ={best_gain:.4f})')

elapsed_comb              = time.time() - t0
selected_rdkit_comb_names = [cols_uncorr[i] for i in selected_rdkit_idx_comb]
n_rdkit_comb              = len(selected_rdkit_comb_names)

print(f'\nОтбор завершён за {elapsed_comb:.1f} с')
print(f'Отобрано RDKit-дескрипторов поверх Morgan FP: {n_rdkit_comb} из {len(cols_uncorr)}')
print(f'CV-R² только Morgan FP : {morgan_only_cv_r2:.4f}')
print(f'CV-R² X_combined       : {best_score_comb:.4f}  (прирост: {best_score_comb - morgan_only_cv_r2:.4f})')
print(f'Признаков в X_combined : 2048 + {n_rdkit_comb} = {2048 + n_rdkit_comb}')
print()
print(f'Отобранные дескрипторы (с учётом FP):')
comb_corr = corr_with_y[selected_rdkit_comb_names].sort_values(ascending=False)
for rank, (name, r_val) in enumerate(comb_corr.items(), 1):
    print(f'  {rank:3d}. {name:<40}  r={r_val:.3f}')

Комбинированный отбор: Morgan FP (2048) фиксированы, добавляем RDKit-дескрипторы...
Кандидатов RDKit: 134
Это займёт 10–30 мин.

Базовый CV-R² (только Morgan FP): 0.2410
  Шаг  1: +MolMR                                   CV-R²=0.3763  (Δ=0.1353)
  Шаг  2: +fr_benzene                              CV-R²=0.3928  (Δ=0.0165)
  Шаг  3: +BCUT2D_CHGLO                            CV-R²=0.4087  (Δ=0.0159)
  Шаг  4: +VSA_EState8                             CV-R²=0.4218  (Δ=0.0131)
  Шаг  5: +AvgIpc                                  CV-R²=0.4298  (Δ=0.0080)
  Шаг  6: +fr_ketone                               CV-R²=0.4344  (Δ=0.0046)
  Шаг  7: +FpDensityMorgan3                        CV-R²=0.4382  (Δ=0.0038)
  Шаг  8: +FpDensityMorgan1                        CV-R²=0.4424  (Δ=0.0042)
  Шаг  9: +fr_Ar_OH                                CV-R²=0.4457  (Δ=0.0033)
  Шаг 10: +fr_COO                                  CV-R²=0.4488  (Δ=0.0031)
  Шаг 11: +fr_nitrile                              CV-R²=0.4519  (Δ=0.

In [13]:
# Сборка X_combined: Morgan FP + K отобранных RDKit-дескрипторов
X_rdkit_comb_raw = X_desc_all[:, selected_rdkit_idx_comb]    # (8572, K), сырые
X_combined       = np.hstack([fp_matrix, X_rdkit_comb_raw])  # (8572, 2048+K)

combined_cols = morgan_cols + selected_rdkit_comb_names
df_combined   = pd.DataFrame(X_combined, columns=combined_cols)
df_combined.to_pickle(OUT_DIR / 'X_combined.pkl')

# Дополняем метаданные
with open(OUT_DIR / 'selected_descriptors.json', encoding='utf-8') as f:
    meta = json.load(f)

meta['rdkit_cols_combined']    = selected_rdkit_comb_names
meta['n_rdkit_combined']       = n_rdkit_comb
meta['n_features_combined']    = int(X_combined.shape[1])
meta['cv_r2_morgan_only']      = float(morgan_only_cv_r2)
meta['cv_r2_combined']         = float(best_score_comb)

with open(OUT_DIR / 'selected_descriptors.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print('Сохранено:')
print(f'  X_combined.pkl — {df_combined.shape[0]} молекул × {df_combined.shape[1]} признаков')
print(f'    Morgan FP  : {N_BITS} бит  (binary, 0/1)')
print(f'    RDKit desc : {n_rdkit_comb} дескрипторов  (raw, не нормализованы)')
print()
print('Итого три набора признаков:')
print(f'  X_morgan.pkl   — (8572, 2048)')
print(f'  X_rdkit.pkl    — (8572, 31)')
print(f'  X_combined.pkl — (8572, {2048 + n_rdkit_comb})')

Сохранено:
  X_combined.pkl — 8572 молекул × 2072 признаков
    Morgan FP  : 2048 бит  (binary, 0/1)
    RDKit desc : 24 дескрипторов  (raw, не нормализованы)

Итого три набора признаков:
  X_morgan.pkl   — (8572, 2048)
  X_rdkit.pkl    — (8572, 31)
  X_combined.pkl — (8572, 2072)


## 10. Baseline: Ridge с кросс-валидацией

Быстрая проверка всех трёх наборов признаков на линейной модели.
Сравниваем:

1. **X_morgan** — только Morgan FP (2048 бит)
2. **X_rdkit** — только 31 отобранный RDKit-дескриптор
3. **X_combined** — Morgan FP + K RDKit-дескрипторов (комбинированный отбор)

Для корректной оценки используется `Pipeline(StandardScaler → Ridge)`, чтобы нормализация не «видела» тест внутри каждого CV-фолда.

> Ridge — линейная модель. 2048 разреженных бит Morgan FP в линейном режиме слабо помогают: каждый бит встречается редко (~0.8%), Ridge не строит взаимодействий. Нелинейные модели в `modeling.ipynb` используют FP эффективнее.

In [14]:
cv5      = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
pipe_reg = lambda: Pipeline([('scaler', StandardScaler()), ('ridge', Ridge(alpha=1.0))])

# 1. X_morgan
scores_morgan = cross_val_score(
    pipe_reg(), fp_matrix, y, cv=cv5, scoring='r2', n_jobs=-1
)

# 2. X_rdkit (31 дескриптор, сырые)
scores_rdkit = cross_val_score(
    pipe_reg(), X_desc_sel_all_raw, y, cv=cv5, scoring='r2', n_jobs=-1
)

# 3. X_combined (Morgan FP + K дескрипторов, сырые)
scores_combined = cross_val_score(
    pipe_reg(), X_combined, y, cv=cv5, scoring='r2', n_jobs=-1
)

print('=' * 65)
print('  BASELINE Ridge + 5-fold CV  (R²)')
print('=' * 65)
print(f'  X_morgan   (2048 FP)                    : {scores_morgan.mean():.4f} ± {scores_morgan.std():.4f}')
print(f'  X_rdkit    (31 дескриптор)               : {scores_rdkit.mean():.4f} ± {scores_rdkit.std():.4f}')
print(f'  X_combined (2048 FP + {n_rdkit_comb} дескрипторов)  : {scores_combined.mean():.4f} ± {scores_combined.std():.4f}')
print('=' * 65)

  BASELINE Ridge + 5-fold CV  (R²)
  X_morgan   (2048 FP)                    : 0.3423 ± 0.0084
  X_rdkit    (31 дескриптор)               : 0.6161 ± 0.0178
  X_combined (2048 FP + 24 дескрипторов)  : 0.5339 ± 0.0191


## Итог

### Воронка отбора дескрипторов

| Этап | Дескрипторов |
|---|---|
| Входных RDKit | 217 |
| После VarianceThreshold | 174 |
| После удаления коллинеарных (\|r\|>0.90) | 134 |
| Отобрано SFS без FP (раздел 7) → **X_rdkit** | **31** |
| Отобрано SFS поверх FP (раздел 9а) → **X_combined** | **2048 + 24** |
| Morgan FP (фиксировано) → **X_morgan** | **2048** |

### Три набора признаков для `modeling.ipynb`

| Файл | Описание | Форма |
|---|---|---|
| `X_morgan.pkl` | Только Morgan FP | (8572, 2048) |
| `X_rdkit.pkl` | Только отобранные RDKit-дескрипторы | (8572, 31) |
| `X_combined.pkl` | Morgan FP + RDKit-дескрипторы (совместный отбор) | (8572, 2048+24) |

Отобранные дескрипторы покрывают все физически значимые типы межмолекулярных взаимодействий, определяющих температуру кипения: дисперсионные/поляризуемость (MolMR, BertzCT), электронные (EState-индексы, парциальные заряды), полярность (TPSA, VSA), водородные связи (NumHAcceptors, fr_NH2), ароматичность (fr_benzene).

**Следующий шаг:** `modeling.ipynb` — каждая модель обучается на всех трёх наборах признаков.